# PDMP Sampler Benchmarks

Standard experiments from the PDMP literature, adapted for the Boomerang & Sticky Boomerang with PLI thinning.

| # | Experiment | Reference | What it measures |
|---|---|---|---|
| 1 | Scaling with n | Bierkens, Fearnhead & Roberts (2019) §6.4 | ESS/grad_eval and ESS/sec vs number of observations |
| 2 | Sparse recovery | Bertazzi & Bierkens (2023) §4.3 | Inclusion probabilities, coefficient recovery |
| 3 | Predictive performance | Goan et al. (2023) Table 1 | NLL, Accuracy, ECE, ESS on real data |

In [ ]:
import os; os.chdir('../..')
import numpy as np
import matplotlib.pyplot as plt
from benchmarks_august.targets.logreg import logreg, logreg_synthetic
from benchmarks_august.samplers import build_sampler, build_kappa, apply_preprocess
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.analysis.metrics import sample_quality, model_performance, _ess, _ess_batch_means

---
## 1. Scaling with n (cf. Bierkens et al. 2019, Fig. 3)

Fix p=10 (dense), vary n. For each n: run Boomerang PLI, measure min-ESS, gradient evals, wall time.
Report ESS/grad_eval and ESS/sec.

In [ ]:
N_skel = 10000
n_resample = 50000
p=10
burnin = 0.2
refresh_rate = 1.0

#n_obs_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
n_obs_list = [100, 500, 1000, 5000]
results_scaling = []

for n_obs in n_obs_list:
    print(f'\n{"="*60}')
    print(f'n_obs = {n_obs}')
    print(f'{"="*60}')
    
    target = logreg_synthetic(n=n_obs, p=p, sparsity='dense', seed=42,
                              prior={'kind': 'gaussian', 'scale': 1.0})
    
    sampler = build_sampler('boomerang', target, N=N_skel, refresh_rate=refresh_rate)
    apply_preprocess(sampler, target, {'method': 'diagonal'})
    sampler.sample_auto(diagnostics=True)
    
    _, x = resample_pdmp_path(sampler, n_samples=n_resample, burnin_frac=burnin)
    ess_per_dim = np.array([_ess_batch_means(x[:, i]) for i in range(p)])
    avg_ess = ess_per_dim.mean()
    #ess_per_sec = avg_ess / wall_seconds
    
    df = sampler.diagnostics_df
    grad_evals = df['rate_evals'].sum()
    wall_sec = df['wall_seconds'].sum()
    
    results_scaling.append({
        'n_obs': n_obs,
        'min_ess': avg_ess.min(),
        'median_ess': np.median(avg_ess),
        'grad_evals': grad_evals,
        'wall_sec': wall_sec,
        'ess_per_grad': avg_ess.min() / grad_evals,
        'ess_per_sec': avg_ess.min() / wall_sec,
    })
    print(f'  min ESS={avg_ess.min():.1f}, grad_evals={grad_evals}, '
          f'ESS/grad={avg_ess.min()/grad_evals:.6f}, ESS/sec={avg_ess.min()/wall_sec:.2f}')

In [ ]:
# Plot scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ns = [r['n_obs'] for r in results_scaling]

ax = axes[0]
ax.plot(ns, [r['ess_per_grad'] for r in results_scaling], 'o-', color='steelblue')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('n (observations)'); ax.set_ylabel('min ESS / gradient eval')
ax.set_title('Computational efficiency')

ax = axes[1]
ax.plot(ns, [r['ess_per_sec'] for r in results_scaling], 'o-', color='darkred')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('n (observations)'); ax.set_ylabel('min ESS / second')
ax.set_title('Wall-clock efficiency')

fig.suptitle('Boomerang: scaling with dataset size (p=10, dense)', fontsize=13)
plt.tight_layout()

---
## 2. Sparse recovery (cf. Bertazzi & Bierkens 2023, §4.3, Fig. 7)

Synthetic sparse logistic regression: p=20, only 3 nonzero coefficients.
Compare Boomerang PLI (continuous) vs Sticky Boomerang PLI (variable selection).
Report: coefficient recovery, marginal inclusion probabilities.

In [ ]:
target_sp = logreg_synthetic(n=500, p=20, sparsity='sparse', seed=42,
                             prior={'kind': 'gaussian', 'scale': 1.0})
print('beta_true:', target_sp.true_params)

In [ ]:
# Boomerang PLI
boom_sp = build_sampler('boomerang_pli', target_sp, N=N_skel, refresh_rate=refresh_rate)
apply_preprocess(boom_sp, target_sp, {'method': 'diagonal'})
boom_sp.sample_auto(diagnostics=True)

# Sticky PLI
kappa_sp = build_kappa({'kind': 'uniform', 'gamma_prior': 0.15}, target_sp)
sticky_sp = build_sampler('sticky_boomerang_pli', target_sp, N=N_skel,
                          kappa=kappa_sp, refresh_rate=refresh_rate)
apply_preprocess(sticky_sp, target_sp, {'method': 'diagonal'})
sticky_sp.sample_auto(diagnostics=True)

In [ ]:
_, x_boom_sp = resample_pdmp_path(boom_sp, n_samples=n_resample, burnin_frac=burnin)
_, x_sticky_sp = resample_sticky_pdmp_path(sticky_sp, n_samples=n_resample, burnin_frac=burnin)

D = target_sp.D
beta_true = target_sp.true_params
inclusion_true = (beta_true != 0).astype(float)
inclusion_sticky = (x_sticky_sp != 0).mean(axis=0)  # marginal inclusion probability

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: coefficient recovery
ax = axes[0]
coords = np.arange(D)
w = 0.25
ax.bar(coords - w, beta_true, width=w, label='truth', color='black', alpha=0.4)
ax.bar(coords, x_boom_sp.mean(axis=0), width=w, label='Boomerang PLI', color='steelblue')
ax.bar(coords + w, x_sticky_sp.mean(axis=0), width=w, label='Sticky PLI', color='coral')
ax.set_xticks(coords[::2])
ax.set_xlabel('coordinate'); ax.set_ylabel('coefficient')
ax.set_title('Coefficient recovery')
ax.legend(fontsize=8)

# Right: inclusion probabilities (sticky only)
ax = axes[1]
ax.bar(coords - 0.15, inclusion_true, width=0.3, label='truly nonzero', color='black', alpha=0.3)
ax.bar(coords + 0.15, inclusion_sticky, width=0.3, label='P(β≠0 | data)', color='coral')
ax.set_xticks(coords[::2])
ax.set_xlabel('coordinate'); ax.set_ylabel('inclusion probability')
ax.set_title('Variable selection (Sticky PLI)')
ax.legend(fontsize=8)

fig.suptitle(f'Sparse logistic regression: p={D}, n=500, 3 nonzero', fontsize=13)
plt.tight_layout()